In [1]:
import json
from pathlib import Path
from pprint import pprint
import statistics


jsonl_path = Path("../output/projects.jsonl")


def iter_entries(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            yield json.loads(line)


entries = list(iter_entries(jsonl_path))

print(f"Loaded {len(entries)} project entries")


# Show first project
project = entries[0]

print("\n=== PROJECT METADATA ===")
pprint(project["project"])


print("\n=== SOURCE FILE COUNT ===")
print(len(project["sources"]))


# -----------------------------
# NEW STATS COMPUTATION
# -----------------------------

file_function_counts = [
    len(source["functions"])
    for source in project["sources"]
]

total_functions = sum(file_function_counts)

avg_functions = statistics.mean(file_function_counts) if file_function_counts else 0
min_functions = min(file_function_counts) if file_function_counts else 0
max_functions = max(file_function_counts) if file_function_counts else 0


print("\n=== FUNCTION STATISTICS ===")
print(f"Total functions in project: {total_functions}")
print(f"Avg functions per file: {avg_functions:.2f}")
print(f"Min functions per file: {min_functions}")
print(f"Max functions per file: {max_functions}")


# Show first few source files
for source in project["sources"][:3]:

    print("\n" + "=" * 80)
    print("FILE:", source["relative_path"])

    print("Functions:", len(source["functions"]))

    for fn in source["functions"][:2]:

        print("\n  FUNCTION:", fn["qualified_name"])
        print("  Signature:", fn["signature"])
        print("  Lines:", f'{fn["start_line"]}-{fn["end_line"]}')

        if fn.get("metrics"):
            print("  LOC:", fn["metrics"]["loc"])
            print("  Tokens:", fn["metrics"]["token_count"])

        if fn.get("decorators"):
            print("  Decorators:", fn["decorators"])

        if fn.get("modifiers"):
            print("  Modifiers:", fn["modifiers"])

        if fn.get("code"):
            print("\n  RAW CODE PREVIEW:")
            print("-" * 40)

            preview = fn["code"]["raw"]
            print(preview)

            if len(fn["code"]["raw"]) > 500:
                print("...")

Loaded 1 project entries

=== PROJECT METADATA ===
{'language': 'python', 'name': 'calc-test', 'repository_url': ''}

=== SOURCE FILE COUNT ===
5

=== FUNCTION STATISTICS ===
Total functions in project: 16
Avg functions per file: 3.20
Min functions per file: 2
Max functions per file: 4

FILE: main.py
Functions: 2

  FUNCTION: run_calculator
  Signature: run_calculator
  Lines: 28-111
  LOC: 84
  Tokens: 251

  RAW CODE PREVIEW:
----------------------------------------
def run_calculator():

    while True:

        print_menu()
        choice = input("Choose an operation: ")

        result = None
        expression = None

        if not validate_choice(choice):
            print("Invalid option\n")
            continue

        # -------------------------
        # BASIC OPERATIONS (1–4)
        # -------------------------
        if choice in ["1", "2", "3", "4"]:

            a = get_number("Enter first number: ")
            b = get_number("Enter second number: ")

            if 